# UTI / urine-culture dataset EDA (run separately per source)

**Do not run until the original file is available and its source-specific target definition is verified.** This notebook does not merge datasets or alter class balance. Change `DATA_PATH`, `TARGET`, and optional `PATIENT_ID` for each independent source. Start with a raw-file copy. Record file version, URL, license and SHA-256 in the data manifest.

The `TARGET` must match the source's actual outcome label. Do not assume a culture-positive label means clinical UTI.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

DATA_PATH = Path('../data/raw/REPLACE_WITH_SOURCE_FILE.csv')
TARGET = 'REPLACE_WITH_VERIFIED_BINARY_TARGET'
PATIENT_ID = None  # set to verified patient identifier, when available
SEED = 271828

if not DATA_PATH.exists():
    raise FileNotFoundError(f'Place the unmodified source file at {DATA_PATH.resolve()}')
df = pd.read_parquet(DATA_PATH) if DATA_PATH.suffix.lower() in {'.parquet', '.pq'} else pd.read_csv(DATA_PATH, low_memory=False)
assert TARGET in df.columns, f'Missing target column {TARGET}'
print('Original rows:', len(df), '| columns:', df.shape[1])
print('Exact duplicate rows:', int(df.duplicated().sum()))
print('Target missing:', int(df[TARGET].isna().sum()))
print('Target counts incl. missing:\n', df[TARGET].value_counts(dropna=False))
print('Data types:\n', df.dtypes.to_string())
print('Missing by column:\n', df.isna().sum().sort_values(ascending=False).to_string())
if PATIENT_ID and PATIENT_ID in df:
    print('Unique patients:', df[PATIENT_ID].nunique(dropna=True), 'repeated-ID rows:', int(df[PATIENT_ID].duplicated(keep=False).sum()))
assert df.loc[df[TARGET].notna()].drop_duplicates().shape[0] >= 2000, 'Reject dataset if fewer than 2,000 valid unique original records.'

## Class balance and missingness
The class counts are source outcomes, not a rebalanced training set. Missingness is displayed without deleting incomplete records.

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
sns.countplot(data=df, x=TARGET, order=df[TARGET].value_counts(dropna=False).index, ax=ax)
ax.set(title='Original class distribution', xlabel=TARGET, ylabel='Rows')
plt.xticks(rotation=25, ha='right'); plt.tight_layout(); plt.show()

missing = df.isna().mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(max(8, len(df.columns)*0.22), 5))
sns.heatmap(df.isna().sample(min(len(df), 2000), random_state=SEED).T, cbar=False, yticklabels=True, ax=ax)
ax.set(title='Missingness heatmap (sampled rows; all columns)', xlabel='sampled records', ylabel='features')
plt.tight_layout(); plt.show()
missing.rename('fraction_missing').to_csv('../results/tables/missingness.csv')

## Numeric and categorical distributions
Review units, impossible values, sparsity and feature availability by outcome before modeling. For many columns, choose and justify a clinical subset for plotting.

In [ ]:
numeric = df.select_dtypes(include=np.number).columns.drop(TARGET, errors='ignore').tolist()
if numeric:
    corr = df[numeric].corr(min_periods=3)
    fig, ax = plt.subplots(figsize=(max(7, len(numeric)*0.5), max(6, len(numeric)*0.45)))
    sns.heatmap(corr, cmap='vlag', center=0, ax=ax)
    ax.set_title('Numeric feature correlation (pairwise complete observations)')
    plt.tight_layout(); plt.show()
    df[numeric].hist(figsize=(14, 10), bins=30); plt.suptitle('Numeric feature distributions'); plt.tight_layout(); plt.show()
    for col in numeric[:12]:
        fig, ax = plt.subplots(figsize=(6,4))
        sns.boxplot(data=df, x=TARGET, y=col, ax=ax)
        ax.set_title(f'{col} by source outcome'); plt.tight_layout(); plt.show()
categorical = [c for c in df.columns if c not in numeric and c != TARGET]
for col in categorical[:12]:
    top = df[col].astype('string').value_counts(dropna=False).head(15).index
    plot_df = df[df[col].astype('string').isin(top)]
    fig, ax = plt.subplots(figsize=(8,4))
    sns.countplot(data=plot_df, y=col, hue=TARGET, order=plot_df[col].value_counts().index, ax=ax)
    ax.set_title(f'{col} distribution by outcome (top levels)'); plt.tight_layout(); plt.show()

## Export source audit facts
The audit CLI also records checksums and identifiers. Do not treat preprocessing, oversampling or bootstrap copies as source observations. For repeated patients, report both sample and patient counts and use grouped splits. Document any exclusions in a separate, versioned cleaning script.